# Machine Learning Para Predição de Desempenho de Alunos do 3º/4º do Ensino Médio

## Integrantes
- Ana Caroline Souza Lira
- Daniel Goulart Camacho Gonçalves
- Fernando Giongo
- Theo Luigi
- João Gabriel Lourenço Marques

---
O projeto tem por objetivo prever os níveis Saeb de proficiência em Lingua Portuguesa (LP) ou Matemática (MT) de alunos do 3º e 4º Anos do Ensino Médio. Para isso, utilizaremos classificadores de Aprendizado de Máquina em cima dos microdados do Saeb de 2023 e a biblioteca Pandas do Python.

# Pré-processamento de dados

In [2]:
# Importando as bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_aluno_original = pd.read_csv('dados/TS_ALUNO_34EM.csv', encoding='latin-1', sep=';')
df_diretor_original = pd.read_csv('dados/TS_DIRETOR.csv', encoding='latin-1', sep=';')
df_escola_original = pd.read_csv('dados/TS_ESCOLA.csv', encoding='latin-1', sep=';')

O próximo passo será definir quais features serão mantidas para análise. Para preservar o escopo do estudo, investigamos o dicionário de dados em busca de variáveis que representassem o contexto socioeconômico familiar, a infraestrutura física das escolas e a capacitação dos professores. Esses fatores foram selecionados primariamente por serem aspectos que, em tese, apresentam grande potencial de influência sobre o desempenho dos alunos. Ao longo do projeto, avaliaremos o comportamento dessas variáveis e sua efetiva relevância com os modelos desenvolvidos.

Nesse momento, também definimos critérios de limpeza dos dados. No conjunto de alunos, foram removidos os registros de estudantes que não compareceram às provas ou que responderam pelo menos três questões do questionário socioeconômico. De forma semelhante, no conjunto de diretores, excluímos os registros daqueles que não preencheram o questionário ou que não eram responsáveis por alunos do 3º ou o 4º ano do Ensino Médio. Por fim, no conjunto de escolas, foram removidas as instituições que não possuíam alunos matriculados nessas séries.

In [3]:
# ================================================================
# Escolhemos a features que temos interesses e as features
# que usaremos para limpar linhas que não tenham dados relevantes
# ================================================================

colunas_de_interesse_aluno = ['ID_ESCOLA', 'ID_ALUNO', 'ID_UF', 'ID_MUNICIPIO', 'ID_AREA', 'IN_PUBLICA', 'ID_SERIE', 'PROFICIENCIA_LP_SAEB',
'PROFICIENCIA_MT_SAEB', 'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03', 'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b',
'TX_RESP_Q05c', 'TX_RESP_Q06','TX_RESP_Q07a','TX_RESP_Q07b','TX_RESP_Q07c','TX_RESP_Q07d','TX_RESP_Q07e','TX_RESP_Q08','TX_RESP_Q09',
'TX_RESP_Q10a','TX_RESP_Q10b','TX_RESP_Q10c','TX_RESP_Q10d','TX_RESP_Q10e','TX_RESP_Q10f','TX_RESP_Q11a','TX_RESP_Q11b','TX_RESP_Q11c',
'TX_RESP_Q12b','TX_RESP_Q12c','TX_RESP_Q14','TX_RESP_Q15b','TX_RESP_Q16','TX_RESP_Q18','TX_RESP_Q21a',
'TX_RESP_Q21b','TX_RESP_Q21c','TX_RESP_Q21d','TX_RESP_Q21e','TX_RESP_Q23d','TX_RESP_Q24']

remover_aluno = {
    'IN_PRESENCA_LP': 0,                    # Remover alunos que não tenham respondido a prova de língua portuguesa
    'IN_PRESENCA_MT': 0,                    # Remover alunos que não tenham respondido a prova de matemática
    'IN_PROFICIENCIA_LP': 0,                # Remover alunos que não tenham proficiência em língua portuguesa
    'IN_PROFICIENCIA_MT': 0,                # Remover alunos que não tenham proficiência em matemática
    'IN_PREENCHIMENTO_QUESTIONARIO': 0}     # Remover alunos que não preencheram o questionário

colunas_de_interesse_diretor = ['ID_ESCOLA', 'ID_SERIE', 'TX_Q020','TX_Q022','TX_Q032','TX_Q033','TX_Q035',
'TX_Q036','TX_Q056','TX_Q057','TX_Q078','TX_Q079','TX_Q081','TX_Q082','TX_Q083', 'TX_Q085','TX_Q087',
'TX_Q108','TX_Q119','TX_Q129','TX_Q130','TX_Q139','TX_Q191','TX_Q194','TX_Q203','TX_Q205','TX_Q206','TX_Q207',
'TX_Q208','TX_Q209']

remover_diretor = {
    'IN_PREENCHIMENTO_QUESTIONARIO': [0],   # Remover diretores que não preencheram o questionário
    'ID_SERIE': [5, 9, 2]}

colunas_de_interesse_escola = ['ID_ESCOLA','PC_FORMACAO_DOCENTE_MEDIO']

remover_escola = {
    'NU_PRESENTES_EM': 0}                   # Remover escolas que não tenham alunos presentes no ensino médio


# ================================================================
# Aplicaremos as regras de limpeza de dados e selecionaremos as features
# ================================================================

print("\n# ================================================================\n")
df_aluno_limpo = df_aluno_original.copy()
df_diretor_limpo = df_diretor_original.copy()
df_escola_limpo = df_escola_original.copy()

for coluna, valor in remover_aluno.items():
    antes = len(df_aluno_limpo)
    df_aluno_limpo = df_aluno_limpo[df_aluno_limpo[coluna] != valor]
    removidas = antes - len(df_aluno_limpo)

    print(
        f"Alunos: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

for coluna, valor in remover_diretor.items():
    antes = len(df_diretor_limpo)
    df_diretor_limpo = df_diretor_limpo[~df_diretor_limpo[coluna].isin(valor)]
    removidas = antes - len(df_diretor_limpo)

    print(
        f"Diretores: removidas {removidas} linhas "
        f"pela regra: {valor} não estar em {coluna}"
    )

for coluna, valor in remover_escola.items():
    antes = len(df_escola_limpo)
    df_escola_limpo = df_escola_limpo[df_escola_limpo[coluna] != valor]
    removidas = antes - len(df_escola_limpo)

    print(
        f"Escolas: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

df_aluno_limpo = df_aluno_limpo[colunas_de_interesse_aluno]
df_diretor_limpo = df_diretor_limpo[colunas_de_interesse_diretor]
df_escola_limpo = df_escola_limpo[colunas_de_interesse_escola]

print("\n# ================================================================\n")
print(f'Alunos: {len(df_aluno_limpo)} linhas')
print(f'Diretores: {len(df_diretor_limpo)} linhas')
print(f'Escolas: {len(df_escola_limpo)} linhas')
print("\n# ================================================================\n")

# Quantidade de linhas removidas do dataset original
print(f'Alunos: {len(df_aluno_original) - len(df_aluno_limpo)} linhas removidas do dataset original -> {(len(df_aluno_limpo))/len(df_aluno_original)*100:.2f}% linhas restantes')
print(f'Diretores: {len(df_diretor_original) - len(df_diretor_limpo)} linhas removidas do dataset original -> {(len(df_diretor_limpo))/len(df_diretor_original)*100:.2f}% linhas restantes')
print(f'Escolas: {len(df_escola_original) - len(df_escola_limpo)} linhas removidas do dataset original -> {(len(df_escola_limpo))/len(df_escola_original)*100:.2f}% linhas restantes')

print("\n# ================================================================\n")


# ================================================================

Alunos: removidas 504829 linhas pela regra: IN_PRESENCA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PRESENCA_MT != 0
Alunos: removidas 2575 linhas pela regra: IN_PROFICIENCIA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PROFICIENCIA_MT != 0
Alunos: removidas 10382 linhas pela regra: IN_PREENCHIMENTO_QUESTIONARIO != 0
Diretores: removidas 17304 linhas pela regra: [0] não estar em IN_PREENCHIMENTO_QUESTIONARIO
Diretores: removidas 72208 linhas pela regra: [5, 9, 2] não estar em ID_SERIE
Escolas: removidas 122 linhas pela regra: NU_PRESENTES_EM != 0

# ================================================================

Alunos: 1372493 linhas
Diretores: 17577 linhas
Escolas: 70029 linhas

# ================================================================

Alunos: 517786 linhas removidas do dataset original -> 72.61% linhas restantes
Diretores: 89512 linhas removidas do dataset original -> 16.41% linhas restantes


In [4]:
# ================================================================
# Podemos ter problemas ao unificar as tabelas se houver escolas
# com o mesmo ID e multiplos diretores
# ================================================================

print("\n# ================================================================\n")
print(f"Alunos em mais de uma escola: {df_aluno_limpo['ID_ALUNO'].duplicated().sum()}")
print(f"Alunos duplicados: {df_aluno_limpo['ID_ALUNO'].duplicated().sum()}")


print(f"Escolas com ID_ESCOLA duplicados: {df_escola_limpo['ID_ESCOLA'].duplicated().sum()}")

escolas_com_diretores_duplicados = (
    df_diretor_limpo['ID_ESCOLA']
    .value_counts()
    .loc[lambda x: x > 1]
)

escolas_sem_diretores = (
    df_escola_limpo['ID_ESCOLA']
    .loc[~df_escola_limpo['ID_ESCOLA'].isin(df_diretor_limpo['ID_ESCOLA'])]
)

print(f'Escolas com mais de um diretor: {len(escolas_com_diretores_duplicados)}')
print("\n# ================================================================\n")


# ================================================================

Alunos em mais de uma escola: 0
Alunos duplicados: 0
Escolas com ID_ESCOLA duplicados: 0
Escolas com mais de um diretor: 723

# ================================================================



In [5]:
# ================================================================
# Vamos tratar as Features transformando de categóricas para numéricas quando possivel
# ================================================================

colunas_diretor_numericas = [
    'TX_Q020',
    'TX_Q022'
]

colunas_diretor_ordinais_AD = [
    'TX_Q056',
    'TX_Q057',
    'TX_Q108'
]

colunas_diretor_ordinais_AE = [
    'TX_Q032',
    'TX_Q033',
    'TX_Q035',
    'TX_Q036',
    'TX_Q191'
]

colunas_diretor_booleanas = [
    'TX_Q078', 'TX_Q079', 'TX_Q081', 'TX_Q082',
    'TX_Q083', 'TX_Q085', 'TX_Q087', 'TX_Q119',
    'TX_Q129', 'TX_Q130', 'TX_Q139', 'TX_Q194',
    'TX_Q203', 'TX_Q205', 'TX_Q206', 'TX_Q207',
    'TX_Q208', 'TX_Q209'
]

colunas_aluno_booleanas = [
    'TX_RESP_Q07a', 'TX_RESP_Q07b', 'TX_RESP_Q07c',
    'TX_RESP_Q07d', 'TX_RESP_Q07e',
    'TX_RESP_Q11a', 'TX_RESP_Q11b', 'TX_RESP_Q11c',
    'TX_RESP_Q15b'
]

colunas_aluno_booleanas_ABC = [
]

colunas_aluno_ordinais_AD = [
    'TX_RESP_Q23d'
]

colunas_aluno_categoricas = [
    'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03',
    'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b',
    'TX_RESP_Q05c', 'TX_RESP_Q06', 'TX_RESP_Q08',
    'TX_RESP_Q09', 'TX_RESP_Q10a', 'TX_RESP_Q10b',
    'TX_RESP_Q10c', 'TX_RESP_Q10d', 'TX_RESP_Q10e',
    'TX_RESP_Q10f', 'TX_RESP_Q12b', 'TX_RESP_Q12c',
    'TX_RESP_Q14', 'TX_RESP_Q16',
    'TX_RESP_Q18', 'TX_RESP_Q21a', 'TX_RESP_Q21b',
    'TX_RESP_Q21c', 'TX_RESP_Q21d', 'TX_RESP_Q21e',
    'TX_RESP_Q24'
]


# ================================================================
# Mapeamentos normalizados [0,1]
# ================================================================

mapa_AD = {
    'A': 0 / 3,
    'B': 1 / 3,
    'C': 2 / 3,
    'D': 3 / 3
}

mapa_AE = {
    'A': 0 / 4,
    'B': 1 / 4,
    'C': 2 / 4,
    'D': 3 / 4,
    'E': 4 / 4
}

mapa_bool = {
    'A': 0.0,
    'B': 1.0
}

mapa_bool_ABC = {
    'A': 0.0,
    'B': 1.0,
    'C': 1.0
}

valores_invalidos = ['*', '.', 'F', '']

# ================================================================
# Conversão das respostas dos diretores
# ================================================================

for col in colunas_diretor_ordinais_AD:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_AD)
            .astype('float32')
        )

for col in colunas_diretor_ordinais_AE:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_AE)
            .astype('float32')
        )

for col in colunas_diretor_booleanas:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_bool)
            .astype('float32')
        )

for col in colunas_aluno_categoricas:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
        )

for col in colunas_aluno_booleanas:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_bool)
            .astype('float32')
        )

for col in colunas_aluno_booleanas_ABC:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_bool_ABC)
            .astype('float32')
        )

for col in colunas_aluno_ordinais_AD:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_AD)
            .astype('float32')
        )

df_aluno_limpo['ID_AREA'] = (
    df_aluno_limpo['ID_AREA']
    .replace({
        1: 0,
        2: 1,
    })
    .astype('float32')
)

df_aluno_limpo['ID_UF'] = df_aluno_limpo['ID_UF'].astype('int8')
df_aluno_limpo['ID_MUNICIPIO'] = df_aluno_limpo['ID_MUNICIPIO'].astype('int32')

# Cada linha restante possui um diretor válido
df_diretor_limpo['POSSUI_DIRETOR'] = np.int8(1)

print("\n# ================================================================\n")
print('Registros duplicados de (ID_ESCOLA, ID_SERIE):',df_diretor_limpo.duplicated(subset=['ID_ESCOLA', 'ID_SERIE']).sum())

# ================================================================
# Estatísticas
# ================================================================

pares_aluno = (
    df_aluno_limpo[['ID_ESCOLA', 'ID_SERIE']]
    .drop_duplicates()
)

pares_diretor = (
    df_diretor_limpo[['ID_ESCOLA', 'ID_SERIE']]
)

pares_com_diretor = (
    pares_aluno.merge(
        pares_diretor,
        on=['ID_ESCOLA', 'ID_SERIE'],
        how='left',
        indicator=True
    )
)

qtd_com = (pares_com_diretor['_merge'] == 'both').sum()
qtd_sem = (pares_com_diretor['_merge'] == 'left_only').sum()

print(
    f'Pares escola-série com diretor: '
    f'{qtd_com:,} -> '
    f'{100*qtd_com/len(pares_com_diretor):.2f}%'
)

print(
    f'Pares escola-série sem diretor: '
    f'{qtd_sem:,} -> '
    f'{100*qtd_sem/len(pares_com_diretor):.2f}%'
)

# ================================================================
# Merge final
# ================================================================

df_diretor_limpo_copy = df_diretor_limpo.copy()
df_escola_limpo_copy = df_escola_limpo.copy()

df_unificado = (
    df_aluno_limpo
    .merge(
        df_diretor_limpo_copy,
        on=['ID_ESCOLA', 'ID_SERIE'],
        how='left'
    )
    .merge(
        df_escola_limpo_copy,
        on='ID_ESCOLA',
        how='left'
    )
)

# Escolas/séries sem diretor recebem 0
df_unificado['POSSUI_DIRETOR'] = (
    df_unificado['POSSUI_DIRETOR']
    .fillna(0)
    .astype(np.int8)
)

registros_com = int(df_unificado['POSSUI_DIRETOR'].sum())
registros_sem = len(df_unificado) - registros_com

pct_com = 100 * registros_com / len(df_unificado)
pct_sem = 100 - pct_com

print(
    f'Registros com diretor: '
    f'{registros_com:,} -> {pct_com:.2f}%'
)

print(
    f'Registros sem diretor: '
    f'{registros_sem:,} -> {pct_sem:.2f}%'
)

print(f'Linhas finais: {len(df_unificado):,}')
print("\n# ================================================================\n")


# ================================================================

Registros duplicados de (ID_ESCOLA, ID_SERIE): 0
Pares escola-série com diretor: 15,447 -> 82.73%
Pares escola-série sem diretor: 3,224 -> 17.27%
Registros com diretor: 1,132,597 -> 82.52%
Registros sem diretor: 239,896 -> 17.48%
Linhas finais: 1,372,493

# ================================================================



In [6]:
# ================================================================
# Adicionando a classificação de nível de acordo com a escala SAEB
# ================================================================
df_final = df_unificado.copy()

df_final = df_final[df_final['PROFICIENCIA_LP_SAEB'] >= 225]
df_final = df_final[df_final['PROFICIENCIA_MT_SAEB'] >= 225]

def classificar_nivel_LP(nota_proficiencia):
    if nota_proficiencia < 300:
        return 1
    elif 300 <= nota_proficiencia < 350:
        return 2
    elif nota_proficiencia >= 350:
        return 3

def classificar_nivel_MT(nota_proficiencia):
    if nota_proficiencia < 300:
        return 1
    elif 300 <= nota_proficiencia < 375:
        return 2
    elif nota_proficiencia >= 375:
        return 3



# Adicionando os níveis de proficiência de cada aluno e removendo as colunas originais de proficiência
df_final['NIVEL_LP'] = df_final['PROFICIENCIA_LP_SAEB'].apply(classificar_nivel_LP)
df_final['NIVEL_MT'] = df_final['PROFICIENCIA_MT_SAEB'].apply(classificar_nivel_MT)
df_final = df_final.drop(columns=['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB'])

df_final = df_final[(df_final['NIVEL_LP'] != 0) & (df_final['NIVEL_MT'] != 0)]

df_final.tail(20)

,ID_ESCOLA,ID_ALUNO,ID_UF,ID_MUNICIPIO,ID_AREA,IN_PUBLICA,ID_SERIE,TX_RESP_Q01,TX_RESP_Q02,TX_RESP_Q03,...,TX_Q203,TX_Q205,TX_Q206,TX_Q207,TX_Q208,TX_Q209,POSSUI_DIRETOR,PC_FORMACAO_DOCENTE_MEDIO,NIVEL_LP,NIVEL_MT
1372458,61436653.0,55701798.0,43,6327057,1.0,1.0,12.0,B,B,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1
1372462,61436653.0,55701805.0,43,6327057,1.0,1.0,12.0,B,C,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,2
1372463,61436653.0,55701806.0,43,6327057,1.0,1.0,12.0,A,C,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,2,1
1372464,61436653.0,55701807.0,43,6327057,1.0,1.0,12.0,A,B,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,2
1372468,61436653.0,55698990.0,43,6327057,1.0,1.0,12.0,B,B,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,3,2
1372470,61436653.0,55698993.0,43,6327057,1.0,1.0,12.0,B,C,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1
1372471,61436653.0,55698995.0,43,6327057,1.0,1.0,12.0,A,C,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1
1372473,61436653.0,55700482.0,43,6327057,1.0,1.0,12.0,A,C,D,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1
1372474,61436653.0,55700483.0,43,6327057,1.0,1.0,12.0,A,C,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1
1372479,61436653.0,55700488.0,43,6327057,1.0,1.0,12.0,B,B,A,...,0.0,1.0,1.0,1.0,0.0,1.0,1,72.6,1,1


In [7]:
from collections import Counter

# Limpar linhas com nulos
df_final = df_final.dropna()

print(Counter(df_final['NIVEL_LP']))
print(Counter(df_final['NIVEL_MT']))

Counter({1: 130586, 2: 105182, 3: 25547})
Counter({1: 161822, 2: 89586, 3: 9907})


In [8]:
# ==========================================
# Preparação dos Dados para Treinamento e Teste
# ==========================================

df_treino = df_final.copy()

# Removendo IDs e variáveis alvo do conjunto do treino
cols_id = ['ID_ESCOLA', 'ID_ALUNO', 'ID_MUNICIPIO']
targets = ['NIVEL_LP', 'NIVEL_MT']
X = df_treino.drop(columns=cols_id + targets).copy()
y_lp = df_treino['NIVEL_LP']
y_mt = df_treino['NIVEL_MT']

# Conversão de colunas categóricas para numéricas
cols_nominal = ['TX_RESP_Q01', 'TX_RESP_Q03', 'TX_RESP_Q04']
cols_ordinal = [
    c for c in X.columns
    if c.startswith('TX_')
    and c not in cols_nominal
    and X[c].dtype == 'object'
]

mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'G': 5, 'H': 6, 'I': 7}
X[cols_ordinal] = X[cols_ordinal].replace(mapping)



/tmp/ipykernel_6108/3653255997.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[cols_ordinal] = X[cols_ordinal].replace(mapping)


# Treinamento dos modelos

### Modelo 1: Regressão Logística

##### Construção, treinamento e métricas

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# Realizando one-hot enconding
X = pd.get_dummies(X, columns=cols_nominal, drop_first=True)
X = pd.get_dummies(X, drop_first=True)


modelo_log = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs'
)

pipeline_log = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', modelo_log)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def rodar_modelo(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # Split 70 / 15 / 15
    # ==================================================

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Cross Validation
    # ==================================================

    cv_results = cross_validate(
        pipeline_log,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:   {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std():.4f})")
    print(f"F1 Macro:   {cv_results['test_f1_macro'].mean():.4f} (+/- {cv_results['test_f1_macro'].std():.4f})")
    print(f"F1 Weighted:{cv_results['test_f1_weighted'].mean():.4f} (+/- {cv_results['test_f1_weighted'].std():.4f})")

    # ==================================================
    # Treino Final (70%)
    # ==================================================

    pipeline_log.fit(X_train, y_train)

    print("\n── PARÂMETROS DO MODELO ──")
    print(pipeline_log.named_steps['modelo'].get_params())

    # ==================================================
    # Validação (15%)
    # ==================================================

    y_val_pred = pipeline_log.predict(X_val)

    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("F1 Macro:", f1_score(y_val, y_val_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted'))
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    # ==================================================
    # Teste final (15%)
    # ==================================================

    y_test_pred = pipeline_log.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))

# Aplicação do modelo
rodar_modelo("NIVEL_LP", y_lp)
rodar_modelo("NIVEL_MT", y_mt)


NIVEL_LP

── 5-Fold CV (TREINO 70%) ──
Accuracy:   0.4675 (+/- 0.0017)
F1 Macro:   0.4138 (+/- 0.0016)
F1 Weighted:0.4734 (+/- 0.0015)

── PARÂMETROS DO MODELO ──
{'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

── VALIDAÇÃO (15%) ──
Accuracy: 0.4646784192667806
F1 Macro: 0.41135686085136913
F1 Weighted: 0.4697487493986243
              precision    recall  f1-score   support

           1       0.64      0.62      0.63     19588
           2       0.45      0.25      0.32     15777
           3       0.19      0.58      0.29      3832

    accuracy                           0.46     39197
   macro avg       0.43      0.48      0.41     39197
weighted avg       0.52      0.46      0.47     39197

[[12115  3984  3489]
 [ 6067  3881  5829]
 [  854  

##### Ajuste de parâmetros

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


def rodar_modelo(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # Split70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Ajuste manual de hiperparâmetros
    # ==================================================
    configs = [
        (1, 'lbfgs'),
        (10,  'lbfgs'),
        (100,  'lbfgs'),
        (1000, 'lbfgs'),
        (1.0,  'saga')
    ]

    melhor_modelo = None
    melhor_f1 = -1

    print("\n── AJUSTE MANUAL (VALIDAÇÃO 15%) ──")

    for C, solver in configs:

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('modelo', LogisticRegression(
                C=C,
                solver=solver,
                max_iter=1000,
                class_weight='balanced'
            ))
        ])

        model.fit(X_train, y_train)
        y_val_pred = model.predict(X_val)

        f1 = f1_score(y_val, y_val_pred, average='weighted')

        print(f"C={C}, solver={solver} → F1 weighted = {f1:.4f}")

        if f1 > melhor_f1:
            melhor_f1 = f1
            melhor_modelo = model

    # ==================================================
    # Teste final (15%)
    # ==================================================
    y_test_pred = melhor_modelo.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))


# Aplicação do modelo
rodar_modelo("NIVEL_LP", y_lp)
rodar_modelo("NIVEL_MT", y_mt)


NIVEL_LP

── AJUSTE MANUAL (VALIDAÇÃO 15%) ──
C=1, solver=lbfgs → F1 weighted = 0.4697
C=10, solver=lbfgs → F1 weighted = 0.4697
C=100, solver=lbfgs → F1 weighted = 0.4697
C=1000, solver=lbfgs → F1 weighted = 0.4697
C=1.0, solver=saga → F1 weighted = 0.4698

── TESTE FINAL (15%) ──
Accuracy: 0.46719220368386144
F1 Macro: 0.41324707973381386
F1 Weighted: 0.47249507580340966
              precision    recall  f1-score   support

           1       0.64      0.62      0.63     19588
           2       0.45      0.25      0.32     15778
           3       0.19      0.57      0.29      3832

    accuracy                           0.47     39198
   macro avg       0.43      0.48      0.41     39198
weighted avg       0.52      0.47      0.47     39198

[[12178  3973  3437]
 [ 6031  3935  5812]
 [  869   763  2200]]

NIVEL_MT

── AJUSTE MANUAL (VALIDAÇÃO 15%) ──
C=1, solver=lbfgs → F1 weighted = 0.5552
C=10, solver=lbfgs → F1 weighted = 0.5552
C=100, solver=lbfgs → F1 weighted = 0.5552
C=100

### Modelo 2: KNN

In [11]:
"""
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def rodar_knn(nome, y):

    print("\n" + "="*70)
    print(nome)
    print("="*70)

    # ==================================================
    # Split 70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Cross Validation no treino (70%)
    # ==================================================
    base_model = KNeighborsClassifier()

    pipeline_base = Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', base_model)
    ])

    cv_results = cross_validate(
        pipeline_base,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:   {cv_results['test_accuracy'].mean():.4f}")
    print(f"F1 Macro:   {cv_results['test_f1_macro'].mean():.4f}")
    print(f"F1 Weighted:{cv_results['test_f1_weighted'].mean():.4f}")

    # ==================================================
    # Ajuste do hiperparâmetro K (15%)
    # ==================================================
    melhores_k = [1, 3, 5, 7, 9, 11, 15, 21]

    melhor_k = None
    melhor_score = -1
    melhor_modelo = None

    print("\n── Ajuste de hiperparâmetro K (VALIDAÇÃO 15%) ──")

    for k in melhores_k:

        modelo = KNeighborsClassifier(n_neighbors=k)

        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('modelo', modelo)
        ])

        pipeline.fit(X_train, y_train)

        y_val_pred = pipeline.predict(X_val)

        score = f1_score(y_val, y_val_pred, average='weighted')

        print(f"k={k} | F1 weighted = {score:.4f}")

        if score > melhor_score:
            melhor_score = score
            melhor_k = k
            melhor_modelo = pipeline

    print(f"\n Melhor k: {melhor_k}")
    print(f" Melhor F1 validação: {melhor_score:.4f}")

    # ==================================================
    # Teste Final (15%)
    # ==================================================
    y_test_pred = melhor_modelo.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))


# Aplicação do modelo
rodar_knn("NIVEL_LP", y_lp)
rodar_knn("NIVEL_MT", y_mt)

"""

'\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix\n\n\nskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n\n\ndef rodar_knn(nome, y):\n\n    print("\n" + "="*70)\n    print(nome)\n    print("="*70)\n\n    # ==================================================\n    # Split 70 / 15 / 15\n    # ==================================================\n    X_train, X_temp, y_train, y_temp = train_test_split(\n        X, y,\n        test_size=0.30,\n        stratify=y,\n        random_state=42\n    )\n\n    X_val, X_test, y_val, y_test = train_test_split(\n        X_temp, y_temp,\n        test_size=0.50,\n        stratify=y_temp,\n        random_state=42\n    )\n\n    # ============================================

### Modelo 3: Árvore de Decisão

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

modelo_tree = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

# Sem StandardScaler — árvore não precisa
pipeline_tree = Pipeline([
    ('modelo', modelo_tree)
])

skf = StratifiedKFold(n_splits=5,
                      shuffle=True,
                      random_state=42)

def rodar_ad(nome, y):
    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # Split 70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Cross Validation
    # ==================================================
    cv_results = cross_validate(
        pipeline_tree,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:    {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std():.4f})")
    print(f"F1 Macro:    {cv_results['test_f1_macro'].mean():.4f} (+/- {cv_results['test_f1_macro'].std():.4f})")
    print(f"F1 Weighted: {cv_results['test_f1_weighted'].mean():.4f} (+/- {cv_results['test_f1_weighted'].std():.4f})")

    # ==================================================
    # Treino Final (70%)
    # ==================================================
    pipeline_tree.fit(X_train, y_train)

    print("\n── PARÂMETROS DO MODELO ──")
    for param, valor in pipeline_tree.named_steps['modelo'].get_params().items():
        print(f"  {param}: {valor}")

    # ==================================================
    # Validação (15%)
    # ==================================================
    y_val_pred = pipeline_tree.predict(X_val)

    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy:  ", accuracy_score(y_val, y_val_pred))
    print("F1 Macro:  ", f1_score(y_val, y_val_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted'))
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    # ==================================================
    # Teste Final (15%)
    # ==================================================
    y_test_pred = pipeline_tree.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:  ", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:  ", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))

# Aplicação do modelo
rodar_ad("NIVEL_LP", y_lp)
rodar_ad("NIVEL_MT", y_mt)



NIVEL_LP

── 5-Fold CV (TREINO 70%) ──
Accuracy:    0.4566 (+/- 0.0014)
F1 Macro:    0.3760 (+/- 0.0018)
F1 Weighted: 0.4586 (+/- 0.0014)

── PARÂMETROS DO MODELO ──
  ccp_alpha: 0.0
  class_weight: balanced
  criterion: gini
  max_depth: None
  max_features: None
  max_leaf_nodes: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  random_state: 42
  splitter: best

── VALIDAÇÃO (15%) ──
Accuracy:   0.4586830624792714
F1 Macro:   0.3774638485141508
F1 Weighted: 0.46043210227170694
              precision    recall  f1-score   support

           1       0.55      0.54      0.55     19588
           2       0.43      0.42      0.42     15777
           3       0.15      0.17      0.16      3832

    accuracy                           0.46     39197
   macro avg       0.38      0.38      0.38     39197
weighted avg       0.46      0.46      0.46     39197

[[10671  7273  1644]
 [ 7168  6664  1945]
 [ 149

### Modelo 4: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def rodar_rf(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # 70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # SKFold no treino (70%)
    # ==================================================
    cv_results = cross_validate(
        modelo_rf,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:   {cv_results['test_accuracy'].mean():.4f}")
    print(f"F1 Macro:   {cv_results['test_f1_macro'].mean():.4f}")
    print(f"F1 Weighted:{cv_results['test_f1_weighted'].mean():.4f}")

    # ==================================================
    # Treino final (70%)
    # ==================================================
    modelo_rf.fit(X_train, y_train)

    # ==================================================
    # Validação (15%)
    # ==================================================
    y_val_pred = modelo_rf.predict(X_val)

    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("F1 Macro:", f1_score(y_val, y_val_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted'))
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    # ==================================================
    # Teste final (15%)
    # ==================================================
    y_test_pred = modelo_rf.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))


# Aplicação do modelo
rodar_rf("NIVEL_LP", y_lp)
rodar_rf("NIVEL_MT", y_mt)


NIVEL_LP

── 5-Fold CV (TREINO 70%) ──
Accuracy:   0.5635
F1 Macro:   0.3894
F1 Weighted:0.5269

── VALIDAÇÃO (15%) ──
Accuracy: 0.5663188509324694
F1 Macro: 0.39374046633523313
F1 Weighted: 0.5303417319167406
              precision    recall  f1-score   support

           1       0.60      0.78      0.68     19588
           2       0.51      0.44      0.47     15777
           3       0.57      0.02      0.04      3832

    accuracy                           0.57     39197
   macro avg       0.56      0.41      0.39     39197
weighted avg       0.56      0.57      0.53     39197

[[15219  4361     8]
 [ 8824  6909    44]
 [ 1424  2338    70]]

── TESTE FINAL (15%) ──
Accuracy: 0.5630134190519924
F1 Macro: 0.38857565219460505
F1 Weighted: 0.5264077483534527
              precision    recall  f1-score   support

           1       0.60      0.78      0.67     19588
           2       0.50      0.43      0.46     15778
           3       0.49      0.01      0.03      3832

    accura

##### Ajuste de hiperparâmetros

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def rodar_rf(nome, y):

    print("\n" + "="*70)
    print(nome)
    print("="*70)

    # ==================================================
    # Split 70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Ajuste de parâmetros manuais
    # ==================================================
    param_grid = [
        {'n_estimators': 100, 'max_depth': None},
        {'n_estimators': 200, 'max_depth': None},
        {'n_estimators': 300, 'max_depth': 10},
        {'n_estimators': 300, 'max_depth': 20},
        {'n_estimators': 400, 'max_depth': None}
    ]

    melhor_modelo = None
    melhor_score = -1
    melhor_params = None

    print("\n── Ajuste manual de parâmetros (VALIDAÇÃO 15%) ──")

    for p in param_grid:

        modelo = RandomForestClassifier(
            n_estimators=p['n_estimators'],
            max_depth=p['max_depth'],
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )

        modelo.fit(X_train, y_train)

        y_val_pred = modelo.predict(X_val)

        score = f1_score(y_val, y_val_pred, average='weighted')

        print(f"{p} | F1 weighted = {score:.4f}")

        if score > melhor_score:
            melhor_score = score
            melhor_modelo = modelo
            melhor_params = p

    print("\n Melhor configuração:", melhor_params)
    print(" Melhor F1 validação:", melhor_score)

    # ==================================================
    # Teste final
    # ==================================================
    y_test_pred = melhor_modelo.predict(X_test)

    print("\n── TESTE FINAL ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))

# Aplicação no modelo
rodar_rf("NIVEL_LP", y_lp)
rodar_rf("NIVEL_MT", y_mt)

## Modelo 5: XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    make_scorer
)

# ================================================================
# XGBoost exige classes 0-indexadas (0, 1, 2, ...).
# Nossos targets estão como 1, 2, 3 -> corrigimos com o offset abaixo.
# Guardamos o offset para reverter na hora de interpretar as predições.
# ================================================================
OFFSET = 1
y_lp_xgb = y_lp - OFFSET
y_mt_xgb = y_mt - OFFSET

# Função principal
def rodar_modelo(nome, y, n_iter=15):
    print("\n" + "=" * 60)
    print(nome)
    print("=" * 60)
    print(f"Distribuição de classes: {dict(zip(*np.unique(y, return_counts=True)))}")

    # Split 70 / 15 / 15
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )

    n_classes = len(np.unique(y_train))

    # ============================================================
    # Espaço de busca para o RandomizedSearchCV.
    # Diferente do ajuste manual (poucas combinações fixas), aqui
    # sorteamos n_iter combinações dentro de faixas continuas/discretas,
    # o que cobre muito mais do espaço de hiperparâmetros.
    # ============================================================
    param_dist = {
        'n_estimators':      [200, 300, 400, 600],
        'max_depth':         [3, 4, 5, 6, 8],
        'learning_rate':     [0.02, 0.05, 0.08, 0.1],
        'subsample':         [0.7, 0.8, 0.9, 1.0],
        'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
        'min_child_weight':  [1, 3, 5, 7],
        'gamma':             [0, 0.1, 0.5, 1],
        'reg_alpha':         [0, 0.01, 0.1],
        'reg_lambda':        [0.5, 1, 2],
    }

    # tree_method='hist' é muito mais rápido que o default em datasets
    # grandes. n_jobs=-1 fica só aqui (no XGBoost); o RandomizedSearchCV
    # roda com n_jobs=1 para não competir por núcleos com o XGBoost --
    # ter os dois em -1 ao mesmo tempo é o que sobrecarrega a máquina.
    base_model = XGBClassifier(
        objective    = 'multi:softprob',
        num_class    = n_classes,
        eval_metric  = 'mlogloss',
        tree_method  = 'hist',
        random_state = 42,
        n_jobs       = -1,
    )

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_macro_scorer = make_scorer(f1_score, average='weighted')

    # ============================================================
    # A busca roda numa SUBAMOSTRA do treino (mais rápido) -- o modelo
    # final é re-treinado no X_train completo com os melhores params.
    # Ajuste frac_busca se ainda estiver lento (ex: 0.1 para 10%).
    # ============================================================
    frac_busca = 0.25
    X_busca = X_train.sample(frac=frac_busca, random_state=42)
    y_busca = y_train.loc[X_busca.index]
    sw_busca = compute_sample_weight('balanced', y_busca)

    search = RandomizedSearchCV(
        estimator           = base_model,
        param_distributions = param_dist,
        n_iter              = n_iter,
        scoring             = f1_macro_scorer,
        cv                  = skf,
        random_state        = 42,
        n_jobs              = 1,
        verbose             = 1,
    )

    print(f"\nBuscando entre {n_iter} combinações de hiperparâmetros "
          f"(3-fold CV, {frac_busca:.0%} do treino = {len(X_busca)} linhas)...")
    search.fit(X_busca, y_busca, sample_weight=sw_busca)

    # sample_weight do treino completo, usado no re-treino final abaixo
    sw_train = compute_sample_weight('balanced', y_train)

    print("\nMelhores parâmetros encontrados:")
    print(search.best_params_)
    print(f"Melhor F1 macro (CV): {search.best_score_:.4f}")

    modelo = search.best_estimator_

    # Re-treino com early stopping usando os melhores parâmetros encontrados
    melhores_params = search.best_params_.copy()
    melhores_params.update(dict(
        objective             = 'multi:softprob',
        num_class             = n_classes,
        eval_metric           = 'mlogloss',
        random_state          = 42,
        n_jobs                = -1,
        early_stopping_rounds = 20,
    ))
    modelo = XGBClassifier(**melhores_params)
    modelo.fit(
        X_train, y_train,
        sample_weight = sw_train,
        eval_set      = [(X_val, y_val)],
        verbose       = False
    )
    print(f"\nMelhor iteração (early stopping): {modelo.best_iteration}")

    # Validação (15%)
    y_val_pred = modelo.predict(X_val)
    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy   :", accuracy_score(y_val, y_val_pred))
    print("F1 Macro   :", f1_score(y_val, y_val_pred, average='macro', zero_division=0))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted', zero_division=0))
    print(classification_report(y_val + OFFSET, y_val_pred + OFFSET, zero_division=0))
    print(confusion_matrix(y_val, y_val_pred))

    # Teste final (15%)
    y_test_pred = modelo.predict(X_test)
    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy   :", accuracy_score(y_test, y_test_pred))
    print("F1 Macro   :", f1_score(y_test, y_test_pred, average='macro', zero_division=0))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted', zero_division=0))
    print(classification_report(y_test + OFFSET, y_test_pred + OFFSET, zero_division=0))
    print(confusion_matrix(y_test, y_test_pred))

    # Feature importance (top 20)
    fi = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\n── TOP 20 FEATURES ──")
    print(fi.head(20).to_string())

    return modelo, fi, search

# Aplicação
modelo_lp, fi_lp, search_lp = rodar_modelo("NIVEL_LP", y_lp_xgb)
modelo_mt, fi_mt, search_mt = rodar_modelo("NIVEL_MT", y_mt_xgb)

### Modelo 6: CatBoost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ==================================================
# MODELO CATBOOST
# ==================================================
modelo_cb = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    class_weights_1 = {
        1: 0.6681,
        2: 0.8266,
        3: 3.4068
    },
    random_state=42,
    verbose=0
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def rodar_catboost(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("CATBOOST")
    print("="*60)

    print("\n── HYPERPARAMETERS ──")
    print(modelo_cb.get_params())

    # ==================================================
    # 70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # SKFold no treino (70%)
    # ==================================================
    cv_results = cross_validate(
        modelo_cb,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:    {cv_results['test_accuracy'].mean():.4f}")
    print(f"F1 Macro:    {cv_results['test_f1_macro'].mean():.4f}")
    print(f"F1 Weighted: {cv_results['test_f1_weighted'].mean():.4f}")

    # ==================================================
    # Treino final (70%)
    # ==================================================
    modelo_cb.fit(X_train, y_train)

    # ==================================================
    # VALIDAÇÃO (15%)
    # ==================================================
    y_val_pred = modelo_cb.predict(X_val)

    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("F1 Macro:", f1_score(y_val, y_val_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted'))
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    # ==================================================
    # TESTE FINAL (15%)
    # ==================================================
    y_test_pred = modelo_cb.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))


# ==================================================
# APLICAÇÃO
# ==================================================
rodar_catboost("NIVEL_LP", y_lp)
rodar_catboost("NIVEL_MT", y_mt)